## Bath_dMFA_2026 Software 6 — Uncertainty and Monte Carlo analysis

So far our dynamic stock model has been **deterministic**: one set of input
numbers in, one answer out. But none of those inputs are known exactly —
registrations are measured with some error, and the average vehicle lifetime
is really a rough figure. We therefore want to ask:

> *If the inputs could plausibly have been a bit different, how different
> would the answer be?*

Actually, we need to go further, because that question assumes that the model
is right, and we only need to adjust the input parameters:

> *If there are **multiple possible models**, whose inputs could plausibly
> have been a bit different, how different would the answer be?*

In this notebook we'll analyse this, in four steps:

1. **Section 1 — Preparing the model.** We'll use the same model as before,
   but wrap it up to easily run multiple times.
2. **Section 2 — Monte Carlo.** Put distributions on the inputs, run the model
   lots of times, and summarise the spread of results as a *fan chart*.
3. **Section 3 — Characterising the uncertainty.** How do we know what
   uncertainty should be assigned to a parameter?  One way is *data-quality
   assessment*.
4. **Section 4 — Model uncertainty.** We'll test how much the answer depends
   on the *choice of model* (e.g. the lifetime distribution, or a different
   physical discard model).

Finally, in Section 5 there are some further exercises on "Monte Carlo
practicalities", if you have time.

### Setup

The same imports and the same input data as in Notebook 2.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats

In [ ]:
DATA = "../data/Bath_dMFA_2026_Software_Data.xlsx"

# Historic inflow of new registrations, 1990-2024 (35 years):
I_c = pd.read_excel(DATA, sheet_name="New registration", index_col=0).values[:, 0]

# Share [%] of [gasoline, BEV] in each year's inflow (we only need the 35 historic years):
BEVshare_cT = pd.read_excel(DATA, sheet_name="BEV_share", index_col=0).values

# Mean lifetime [yr] of [gasoline, BEV]:
lifetime0 = pd.read_excel(DATA, sheet_name="Lifetime", index_col=0).values[0].astype(float)

years = np.arange(1990, 2025)  # the 35 historic years, handy for plotting

print("inflow      I_c          ", I_c.shape)
print("BEV share   BEVshare_cT  ", BEVshare_cT.shape)
print("lifetime    lifetime0    ", lifetime0, "(gasoline, BEV)")

### Prep: the model as a function

In the previous Notebook 2, the steps of the model calculation were spread out across multiple cells.  This was good for understanding what was happening, but inconvenient for uncertainty analysis, when we need to run the model many times with different inputs.

As a first step, we'll wrap the inflow-driven model into a single pure function -- that is, a function whose output depends only on the parameter values given explicitly.  This means, for example, that it does not read any external data files within the function (otherwise it might give different output for the same inputs, if those external files changed).

Nothing here is new — it is the same three-step calculation (outflow from the discard pdf, stock change from the balance, stock as a cumulative sum) — just packaged so we can call it with different inputs. The uncertain inputs are its three arguments: `inflow`, `bev_share` and `lifetime`.

In [ ]:
def run_inflow_driven_model(inflow, bev_share, lifetime, sigma_frac=0.3):
    """Inflow-driven dynamic stock model (Notebook 2), as a callable function.

    Parameters
    ----------
    inflow     : array (35,)    new registrations per year, 1990-2024
    bev_share  : array (35, 2)  share [%] of [gasoline, BEV] in each year's inflow
    lifetime   : array (2,)     mean lifetime [yr] of [gasoline, BEV]
    sigma_frac : float          std-dev of the discard pdf, as a fraction of the mean

    Returns
    -------
    S_tcT, O_tcT : arrays (35, 35, 2)   stock and outflow by year t, age-cohort c, technology T
    """
    n = len(inflow)                                  # 35 historic years
    lifetime = np.asarray(lifetime, float).reshape(2, 1)

    # Split the inflow into the two drive technologies (as in Notebook 2):
    I_cT = np.einsum("c,cT->cT", inflow, bev_share) / 100

    # Normally-distributed discard probability, mean = lifetime, sd = sigma_frac * lifetime:
    pdf = scipy.stats.norm.pdf(np.arange(0, n), lifetime, sigma_frac * lifetime)  # (2, n)

    S_tcT = np.zeros((n, n, 2))
    DS_tcT = np.zeros((n, n, 2))
    O_tcT = np.zeros((n, n, 2))

    for drivetech in range(2):                       # for both drive technologies
        for year in range(n):                        # for all historic years
            veleave = np.zeros(n)                    # vehicles leaving this year, by age-cohort
            for age_cohort in range(year + 1):       # for all age-cohorts up to this year
                veleave[age_cohort] = I_cT[age_cohort, drivetech] * pdf[drivetech, year - age_cohort]
            O_tcT[year, :, drivetech] += veleave                 # 1. outflow from the pdf
            DS_tcT[year, year, drivetech] = I_cT[year, drivetech]  # 2. stock change = inflow ...
            DS_tcT[year, :, drivetech] -= O_tcT[year, :, drivetech]  #    ... minus outflow

    S_tcT = np.cumsum(DS_tcT, axis=0)                # 3. stock = cumulative stock change
    return S_tcT, O_tcT


**Check it reproduces the previous Notebook 2.** We'll run the function on the
original input data, and we expect it to give back the same numbers as the
original notebook. Check the results below for the 2022 fleet of size and
total historic outflow against your previous results.

In [ ]:
S_tcT, O_tcT = run_inflow_driven_model(I_c, BEVshare_cT[0:35, :], lifetime0)

O_det = O_tcT.sum(axis=(1, 2))   # deterministic total outflow per year  (our reference line)
S_det = S_tcT.sum(axis=(1, 2))   # deterministic total stock per year

print(f"Stock in 2022:  {S_tcT[32].sum():>14,.0f}")
print(f"Total outflow:  {O_tcT.sum():>14,.0f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(years, S_det); ax[0].set_title("Total stock (fleet size)")
ax[1].plot(years, O_det); ax[1].set_title("Total outflow (end-of-life vehicles)")
for a in ax:
    a.set_xlabel("year"); a.set_ylabel("vehicles")
fig.tight_layout()

In [ ]:
CHECKED_MODEL_OUTPUT = False
assert CHECKED_MODEL_OUTPUT, "set to True after you have checked"

## Section 2 — Monte Carlo: propagating input uncertainty

Now we're ready to use this model-function for uncertainty analysis.
The idea of **Monte Carlo** is simple:

1. Describe each uncertain input with a *distribution* rather than a single number.
2. Draw a random set of inputs from those distributions and run the model.
3. Repeat many times, and look at the *spread* of the results.

For now we won't worry about how to choose the input distributions, just to get the machinery working. In Section 3 below we will see how we can replace them with values from a data-quality assessment.

| input | initial uncertainty estimate |
|---|---|
| inflow (registrations) | ±5 % each year, independent |
| BEV share | ±10 % each year, independent |
| mean lifetime | ±5 % |

(We treat each year's inflow error as *independent* here, and assume that the mean lifetimes apply to all cohorts equally — but these are modelling choices we can revisit later.)

### 2.1 What does one random set of inputs look like?

We use a NumPy random generator with a fixed seed so the notebook is reproducible: re-running
it gives the same "random" numbers. Below we draw one perturbed inflow series and compare it
to the data.

In [ ]:
rng = np.random.default_rng(20260701)  # the number doesn't matter, here I've used the date

# CV stands for "coefficient of variation": the standard deviation
# normalised by the mean value.

CV_INFLOW = 0.05    # placeholder: 5% spread on each year's registrations
CV_BEVSHARE = 0.10  # placeholder: 10% spread on the BEV share
CV_LIFETIME = 0.05  # placeholder: 5% spread on mean lifetime  (deliberately revisited in §3)

inflow_sample = rng.normal(I_c, CV_INFLOW * I_c)

plt.figure(figsize=(8, 4))
plt.plot(years, I_c, "k-", label="data (best guess)")
plt.plot(years, inflow_sample, "C1.-", label="one random sample")
plt.title("Inflow: data vs. one Monte Carlo sample")
plt.xlabel("year"); plt.ylabel("new registrations / yr"); plt.legend();

> **Exercise 1**: try rerunning the cell above.  Does the orange random sample change?  Why?  What happens if you change the seed?
>
> (the shortcut `Run > Run Selected Cell and Do Not Advance` is useful to visually see the plot changing without the notebook scrolling down when you run the cell)

> **Exercise 2**: try increasing uncertainty by adjusting `CV_INFLOW`.  If you
> make it large enough, you may see negative values which are not physically
> meaningful.  This can be solved by choosing a different distribution shape
> (e.g. lognormal), by truncating (e.g. rejecting negative samples), or by
> clipping.  Try adding clipping to ensure your samples are meaningful.
>
> (See the function `np.clip`.  In JupyterLab you can put the cursor inside
> the brackets and press *Shift-Tab* for the signature, or run `rng.clip?`
> in a cell, or see the
> [online docs](https://numpy.org/doc/stable/reference/generated/numpy.clip.html#numpy.clip))

To run the full model we need a random draw of *all three* inputs at once.
The helper function `draw_sample` below packages that up; we've written the
structure and left the three random draws (the `...`) for you to fill in.

**Your turn:** replace each `...` so the input is drawn from a normal
distribution centred on its best-guess value, with a standard deviation
of `CV * mean`. You saw the pattern for the inflow in 2.1 just above.

*Help with the random sampling functions:*  As above, in JupyterLab you can
start writing the package name (like `rng.`) and press *Tab* to see the
options. You can then write a function call (like `rng.normal(`) and press
*Shift-Tab* for the signature, or run `rng.normal?` in a cell.  Or see the
[online docs](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.normal.html).

In [ ]:
def draw_sample(rng):
    """Draw one random (inflow, bev_share, lifetime) set from the distributions above."""
    # inflow: independent normal noise around each year's value, kept non-negative
    inflow = ...

    # BEV share: perturb the BEV column, keep it within [0, 100];
    # gasoline takes the rest
    bev = BEVshare_cT[0:35, 1]
    bev = ...
    bev_share = np.column_stack([100 - bev, bev])

    # mean lifetime of [gasoline, BEV], kept positive (minimum 1)
    lifetime = ...
    return inflow, bev_share, lifetime

**Check your work:** run the cell below a few times — once your function is
complete, it should run without errors, and, the numbers should be
*different* each run (that's the randomness). Switch `OUTPUT_INDEX`
between `0`, `1` and `2` to inspect the inflow, BEV share or lifetime.

Make sure the BEV sampled share values physically sensible (clipped to 0–100%,
with the gasoline share making up the remainder).

If you see an error like "unsupported operand type(s) for -: 'int' and
'ellipsis'", that means you've not yet replaced the `...` placeholder with
real code.

In [ ]:
OUTPUT_INDEX = 2  # 0=inflow, 1=bev_share, 2=lifetime

for _ in range(5):
    print(draw_sample(rng)[OUTPUT_INDEX][:5])

> **Optional exercise — why share one `bev` sample?**
>
> 2. In `draw_sample` the gasoline share is computed as `100 - bev`,
> so the two always add up to 100 %. What if you instead sampled the
> gasoline share *independently* (its own `rng.normal` around `100 - bev`)?
> Do the two shares still add up to 100 %?

### 2.2 A handful of runs

Before running hundreds, let's overlay the total outflow from just **8** random input sets,
to build intuition. Each thin line is one model run; the dashed line is the deterministic
(best-guess) result from the Prep section.

**Predict first:** by 2024, how wide do you expect the spread of total outflow to be —
a couple of %, 10 %, more? Jot down a guess before you run it.

In [ ]:
plt.figure(figsize=(8, 5))
for _ in range(8):
    inflow, bev_share, lifetime = draw_sample(rng)
    S_s, O_s = run_inflow_driven_model(inflow, bev_share, lifetime)
    plt.plot(years, O_s.sum(axis=(1, 2)), color="C0", alpha=0.4)
plt.plot(years, O_det, "k--", lw=2, label="deterministic")
plt.title("Total outflow — 8 random input sets")
plt.xlabel("year"); plt.ylabel("vehicles / yr"); plt.legend();

Does this show a similar uncertainty range to what you expected?
Go back and adjust the `CV_...` values and re-run this to see how it
affects the results.  What happens if you increase or decrease the number
of samples drawn from the default 8?

### 2.3 A full Monte Carlo ensemble

That's useful to see how the uncertain model behaves, but we're not yet storing
any results, and we need more samples to get stable estimates.  Now we'll scale
up to use more samples.

Draw `N = 1000` input sets, run the model on each, and collect the **total
outflow per year** into an array `O_samples` of shape `(N, 35)`. Similarly
collect the total stock into an array `S_samples`.

*Hint:* loop `for i in range(N):`, call `draw_sample(rng)` and
`run_inflow_driven_model(...)`, and store `O_s.sum(axis=(1, 2))` into row `i`.

*Hint:* create the empty array using the `np.zeros()` function.

In [ ]:
N = 1000
O_samples = ...   # an (N, 35) array: total outflow per year, one row per sample
S_samples = ...   # an (N, 35) array: total stock  per year, one row per sample

for i in range(N):
    ...           # draw inputs, run the model, store the per-year totals in row i

### 2.4 Do the numbers make sense?

Before plotting anything, it's worth a quick look at the raw `O_samples`
array.  For example, check the shape is as expected — 1000 rows (samples)
by 35 columns (years); look at the first few samples for the most recent years;
and check that the mean across the samples should is similar to the
deterministic line from the Prep section.

### 2.5 The spread in a single year

Each *column* of `O_samples` holds the 1000 possible values of the outflow in one year. A histogram shows the shape of that distribution. Let's compare two years: 2005, when the outflow was rising steeply, and 2024, the most recent year.

In [ ]:
t_2005 = 2005 - 1990   # column index for 2005
t_2024 = 2024 - 1990   # column index for 2024

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
ax[0].hist(O_samples[:, t_2005], bins=30, color="C0", alpha=0.7)
ax[0].set_title("Total outflow in 2005")
ax[1].hist(O_samples[:, t_2024], bins=30, color="C0", alpha=0.7)
ax[1].set_title("Total outflow in 2024")
for a in ax:
    a.set_xlabel("vehicles / yr"); a.set_ylabel("number of samples")
fig.tight_layout();

The 2005 distribution is much wider, in absolute terms, than the 2024 one.
Why is the outflow most uncertain while it is rising fastest? *(Hint: the
dominant source of spread here is the lifetime, which shifts whole cohorts
earlier or later in time — and that moves the steeply-rising part of the curve
left and right.)*

> **Optional exercise:** Try the same histograms for the stock, `S_samples`.

### 2.6 Summarising a distribution with percentiles

We don't want a histogram for all 35 years. Instead we summarise each year's
distribution with a few **percentiles**: the *p*-th percentile is the value
below which *p* % of the samples fall. The 50th is the median; the 5th and
95th bracket the central 90 % of the samples.

**Your turn:** work out the single `np.percentile` call that returns the 5th, 25th, 50th, 75th and 95th percentiles of the 2024 outflow — the column `O_samples[:, t_2024]`. Print it, and check the numbers against the 2024 histogram above.

*Hint:* `np.percentile` takes the data and a list of percentiles. Shift-Tab inside the brackets for the arguments, or see https://numpy.org/doc/stable/reference/generated/numpy.percentile.html

In [ ]:
# The 5th, 25th, 50th, 75th and 95th percentiles of the 2024 outflow:
pct_2024 = ...
print(pct_2024)

Your five numbers (`pct_2024`) are just markers on the distribution we plotted. Here they are drawn on top of the 2024 histogram — the shaded band is the 5–95 % range, the dashed line is the median:

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(O_samples[:, t_2024], bins=30, color="C0", alpha=0.6)
plt.axvspan(pct_2024[0], pct_2024[4], color="C1", alpha=0.15, label="5–95 %")
plt.axvline(pct_2024[2], color="C1", lw=2, ls="--", label="median")
plt.title("Total outflow in 2024, with percentile markers")
plt.xlabel("vehicles / yr"); plt.ylabel("number of samples"); plt.legend();

### 2.7 The fan chart: percentiles across every year

Now the same thing for *all* years at once. Passing `axis=0` makes
`np.percentile` work down the samples axis (the rows), returning an array of
shape `(5, 35)`: one row per percentile, one column per year. So `pct[2]` is
the median in every year, `pct[0]` and `pct[4]` the 5th and 95th, and so on.
The **fan chart** simply shades between these bands, which is more readable
than 1000 lines.

**Your turn:** fill in (1) the `pct = ...` line — the percentiles across the samples (which `axis`?) — and (2) the band edges in the plot, using the rows of `pct` (`pct[0]` is the 5th percentile … `pct[4]` the 95th). The rest of the plotting is written for you.

In [ ]:
# 1. The 5/25/50/75/95th percentiles of every year at once (aim for shape (5, 35)):
pct = ...
print(pct.shape)

# 2. Fill in the band edges from the rows of `pct`; the rest is written for you:
plt.figure(figsize=(8, 5))
plt.fill_between(years, ..., ..., color="C0", alpha=0.20, label="5–95 %")
plt.fill_between(years, ..., ..., color="C0", alpha=0.35, label="25–75 %")
plt.plot(years, ..., color="C0", lw=2, label="median")
plt.plot(years, O_det, "k--", lw=2, label="deterministic")
plt.title("Total outflow of end-of-life vehicles, EUR20 — Monte Carlo fan chart")
plt.xlabel("year"); plt.ylabel("vehicles / yr"); plt.legend();

**Specific numeric ranges:** how wide is the 5–95 % band in 2024, as a
fraction of the median? Fill in `rel_2024` below. Compare it with the guess
you made earlier.

In [ ]:
rel_2024 = ...   # width of the 5-95% band in 2024, as a fraction of the median
print(f"2024 outflow: median {pct[2, -1]:,.0f},  5-95% band is ±{0.5 * rel_2024 * 100:.1f}% of the median")

### Summary

We can now take a set of input uncertainties and turn it into a band of
plausible results. Right now those input spreads are just initial estimates,
so the width of the fan is not yet necessarily meaningful. The next section
addresses how we determine the input spreads for different datasets.

## Section 3 — Characterising the uncertainty

In Section 2 we picked the input spreads (±5 %, ±10 %, ±5 %) out of thin air.
The fan is only meaningful if those spreads reflect how good each data source
actually is. A **data-quality assessment** is a systematic way to turn what we
know about a source — where it comes from, how complete it is, how well it
matches what we need — into a number we can use. We'll use a *pedigree-matrix*
approach (after Laner et al., 2016). 

### 3.1 A pedigree matrix

The pedigree approach scores each source on a handful of **indicators**, each
from 1 (best) to 4 (worst):

- `Reliability` — Focus on the data source: documentation of data generation,
  e.g., assessment of sampling method, verification methods, reviewing
  processes.
- `Completeness` — Composition of the date of all relevant mass flows. Possible
  over- or underestimation is assessed.
- `Temporal correlation` — Congruence of the available date and the ideal date
  with respect to time reference.
- `Geographical correlation` — Congruence of the available date and the ideal
  date with respect to geographical reference.
- `Other correlation` — Congruence of the available date and the ideal date with
  respect to technology, product, etc.

Each step away from "1" adds some uncertainty; we combine the indicators into a
single coefficient of variation. The exact numerical uncertainty associated with
each qualitative judgement is somewhat arbitrary; however, the systematic
procedure allows for the relative importance of different uncertainty sources to
be judged, and gives a rough indication of the level of uncertainty associated
with different data. In the end, when uncertainty in results is driven by
certain uncertain inputs, the uncertainty associated with those inputs should be
subject to further specific consideration.

The code below implements the Laner et al CV calculation for you.

In [ ]:
# Pedigree-to-uncertainty calculation following Laner et al. (2016), Table 2.
#
# Each indicator score (1 = best ... 4 = worst) is turned into a coefficient of
# variation [%] with an exponential function. For completeness, temporal,
# geographical and other correlation:
#
#     score == 1 -> CV = 0
#     score in 1..4 -> CV[%] = A * exp(B * (score - 1))            (Laner eq. 1)
#
# Reliability uses a shifted version, because even a perfectly documented source
# still carries some uncertainty (so its best score is not zero):
#
#     score in 1..4 -> CV[%] = A * exp(B * score)                  (Laner eq. 2)
#
# B = 1.105 throughout; A scales with how sensitive the flow is to the
# indicator. Laner gives A = 0.375 (not sensitive), 0.75 (medium) and 1.5
# (highly sensitive); here we use the medium-sensitivity value, which reproduces
# the reliability row and the "medium sensitive" row of Laner's Table 2.
B_LANER = 1.105
A_LANER = 0.75  # medium sensitivity


def _indicator_cv(score, reliability=False):
    """CV (as a fraction) for one indicator score, per Laner et al. (2016)."""
    if reliability:
        cv_percent = A_LANER * np.exp(B_LANER * score)        # eq. 2: no zero floor
    elif score == 1:
        cv_percent = 0.0                                      # eq. 1: best score -> 0
    else:
        cv_percent = A_LANER * np.exp(B_LANER * (score - 1))  # eq. 1
    return cv_percent / 100  # Laner's table is in %, but we work with fractions


def cv_from_scores(scores):
    """Combine five indicator scores into one coefficient of variation.

    `scores` is [reliability, completeness, temporal correlation, geographical
    correlation, other correlation], each from 1 (best) to 4 (worst). The five
    indicator CVs are combined in quadrature (root-sum-square, Laner eq. 3).
    """
    reliability, *correlations = scores
    cvs = [_indicator_cv(reliability, reliability=True)]
    cvs += [_indicator_cv(s) for s in correlations]
    return np.sqrt(sum(cv ** 2 for cv in cvs))

Test it out: what CV do you get for different scopes?

In [ ]:
# The order is `[reliability, completeness, temporal correlation, geographical
# correlation, other correlation]`
cv_from_scores([...])   # replace `...` with actual scores from 1-4

 ### 3.2 Data quality assessment for this model

Our three inputs come from quite different places:

- Inflow — new car registrations.
- BEV share.
- Mean lifetime.

> **Exercise:** Review the information in the data spreadsheet about the
    provenance of each dataset. Rank the three inputs from most to least
    trustworthy. Which do you think should have the widest uncertainty
    distribution? And — separately — which input's uncertainty do you think will
    matter *most* for the outflow result? 

**Your turn:** Score each source on the five indicators. This requires some judgement/assumptions.

In [ ]:
scores = {
    "inflow":    ...,   # [reliability, completeness, temporal, geographical, other]
    "bev_share": ...,
    "lifetime":  ...,
}

Turn the scores into coefficients of variation:

In [ ]:
for name, sc in scores.items():
    print(f"{name:10s}  scores {sc}  ->  CV = {cv_from_scores(sc):4.0%}")

CV_INFLOW_3 = cv_from_scores(scores["inflow"])
CV_BEVSHARE_3 = cv_from_scores(scores["bev_share"])
CV_LIFETIME_3 = cv_from_scores(scores["lifetime"])

### 3.3 Re-run the Monte Carlo with the characterised spreads

Now we feed those CVs into exactly the same machinery as Section 2. The cell
below adopts the new spreads and re-runs the ensemble, re-using your
`draw_sample` and the model. We keep the Section 2 result, `O_samples`, so we
can compare the two.

In [ ]:
# Adopt the characterised spreads. draw_sample reads these module-level CVs, so re-running it
# now uses the new values. (If you go back and re-run Section 2, reset these to 0.05/0.10/0.05.)
CV_INFLOW = CV_INFLOW_3
CV_BEVSHARE = CV_BEVSHARE_3
CV_LIFETIME = CV_LIFETIME_3

O_samples_3 = np.zeros((N, 35))
for i in range(N):
    # draw one random set of inputs from the characterised distributions
    inflow, bev_share, lifetime = draw_sample(rng)
    # run the model on them
    S_s, O_s = run_inflow_driven_model(inflow, bev_share, lifetime)
    # store this run's total outflow per year in row i
    O_samples_3[i] = O_s.sum(axis=(1, 2))

print("done:", O_samples_3.shape)

**Your turn:** draw the Section 2 fan and the new Section 3 fan on the same
axes, so you can see how characterising the data changed the picture. Copy the
code already used above — here you need it for *both* `O_samples` and
`O_samples_3`. 

In [ ]:
...

You should see that the new fan is much wider — the extra width comes from
the lifetime's large CV.

## Section 4 — Model uncertainty

Sections 2 and 3 propagated uncertainty in the input *data*. But we also made
*modelling choices* that the Monte Carlo above never questioned: the discard
probability is a *normal* distribution, its width is 30 % of the mean, and —
more fundamentally — that we chose a *cohort-survival* model at all. These are
**structural** (or **model**) uncertainties. They are harder to put a number on
than data uncertainty, but they can matter just as much.

In this section we'll test two modelling alternatives. 

### 4.1 The width of the lifetime distribution

`run_inflow_driven_model` has a `sigma_frac` argument: the standard deviation of
the discard curve as a fraction of the mean lifetime (default 0.3). A narrow
curve (0.1) discards a cohort within a few years of its mean age; a wide one
(0.7) smears discards over a long period. (We already saw these different models
in Notebook 2.)

**Your turn:** run the model on the unperturbed (deterministic / original) data
for `sigma_frac` of 0.1, 0.3 and 0.7, and plot the total outflow of each on the
one chart. How does the width change the shape of the outflow curve? 

In [ ]:
plt.figure(figsize=(8, 5))
for sf in [0.1, 0.3, 0.7]:
    ...   # run the model with sigma_frac=sf, then plot its total outflow vs years, labelled by sf
plt.title("Total outflow for different lifetime-distribution widths")
plt.xlabel("year"); plt.ylabel("vehicles / yr"); plt.legend();

> **Optional exercise:** Adapt your `draw_sample` function to also return
    sampled values for `sigma_frac`, and re-run the Monte Carlo simulation from
    Section 2.3 above. This will show the *overall* uncertainty considering both
    the input data we looked at first, together with uncertainty about the right
    model to pick. To do this, you'll have to make an assumption about the
    probability distribution for different sigma values. Then, plot the fan
    chart again using these samples.

> **Optional exercise 2:** This combines all uncertainty into one spread of
    output parameters. Instead, you might want to show the three fan charts
    corresponding to different fixed values of `sigma_frac`, with uncertain
    samples of the input parameters as before. This avoids committing to an
    assumption about the relative probability of different sigma values, leaving
    that to the reader to judge for themselves.

### 4.2 A different discard model: leaching

The cohort-survival model tracks every age-cohort and discards it according to
the lifetime curve. A much simpler alternative is a **leaching** (first-order)
model: a *fixed fraction* of the stock leaves every year, `k = 1 / mean
lifetime`, regardless of age. There are no cohorts — the model has no memory of
when a vehicle entered.

This is clearly less realistic for cars (a brand-new car is treated as just as
likely to be scrapped as a 20-year-old one), but it illustrates that model
uncertainty can involve more significant changes in the model.
The balance each year is:

> outflow &nbsp; `O(t) = k · S(t−1)` &nbsp;&nbsp; and &nbsp;&nbsp; stock &nbsp; `S(t) = S(t−1) + I(t) − O(t)`

**Your turn:** complete the two physics lines in the loop.

In [ ]:
def run_leaching_model(inflow, bev_share, lifetime):
    """First-order 'leaching' model: a fixed fraction k = 1/lifetime of the stock leaves each year.
    Returns total stock and outflow per year and technology, each shape (n, 2)."""
    n = len(inflow)
    I_cT = np.einsum("c,cT->cT", inflow, bev_share) / 100   # inflow by technology, (n, 2)
    k = 1.0 / np.asarray(lifetime, float)                    # leaching rate per technology, (2,)
    S = np.zeros((n, 2))
    O = np.zeros((n, 2))
    for t in range(n):
        prev = S[t - 1] if t > 0 else np.zeros(2)
        O[t] = ...        # outflow: a fraction k of last year's stock
        S[t] = ...        # balance: previous stock + this year's inflow - this year's outflow
    return S, O

Compare the two models on the same best-guess inputs:

In [ ]:
S_leach, O_leach = run_leaching_model(I_c, BEVshare_cT[0:35, :], lifetime0)

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
ax[0].plot(years, S_det, label="cohort-survival")
ax[0].plot(years, S_leach.sum(1), label="leaching")
ax[0].set_title("Total stock"); ax[0].legend()
ax[1].plot(years, O_det, label="cohort-survival")
ax[1].plot(years, O_leach.sum(1), label="leaching")
ax[1].set_title("Total outflow"); ax[1].legend()
for a in ax:
    a.set_xlabel("year"); a.set_ylabel("vehicles")
fig.tight_layout();

### 4.3 Model vs data uncertainty

So how big is this *structural* difference, compared with the *data* uncertainty
we characterised in Section 3? We can put the leaching result on top of the
Section 3 fan and see whether it falls inside or outside the band of "plausible,
given the data". 

In [ ]:
band = np.percentile(O_samples_3, [5, 95], axis=0)

plt.figure(figsize=(8, 5))
plt.fill_between(years, band[0], band[1], color="C3", alpha=0.25,
                 label="Section 3 data-uncertainty band (5–95 %)")
plt.plot(years, O_det, "C0", lw=2, label="cohort-survival (best guess)")
plt.plot(years, O_leach.sum(1), "C1", lw=2, label="leaching (best guess)")
plt.title("Model-structure difference vs the data-uncertainty band")
plt.xlabel("year"); plt.ylabel("vehicles / yr"); plt.legend();

We should see that in the years where the leaching curve falls *outside* the
data band, the choice of model matters more than all the data uncertainty
combined — and no amount of better data would reveal it, because it is a
question of model structure, not measurement. That is why it can be useful to
report structural / scenario uncertainty (e.g. by showing results from more than
one model) alongside a Monte Carlo band of quantified parameter uncertainty.

> **Optional exercise:** Extend this plot to show *two* fans, by drawing Monte
Carlo samples for the leaching model.

### Conclusions

In this notebook, we have added an uncertainty analysis on top of the Notebook 2
historical stock model, which:

- **propagates** input uncertainty by Monte Carlo (Section 2),
- **characterises** the appropriate input uncertainty from the data sources (Section 3), and
- **tests uncertainty about model structure** with a different discard model (Section 4).